In [1]:
import copy
import numpy as np
from typing import List, Dict, Tuple
from collections import OrderedDict

In [3]:
# Weighted Federated Average

def federated_average(
    client_weights: List[Dict[str, np.ndarray]],
    client_sizes: List[int]
) -> Dict[str, np.ndarray]:
    """
    Aggregate client model weights using weighted FedAvg.

    Args:
        client_weights : list of state_dicts (numpy arrays) from each client
        client_sizes   : number of training samples per client

    Returns:
        Aggregated global model weights (same structure as input dicts)
    """
    total_samples = sum(client_sizes)
    scaling_factors = [n / total_samples for n in client_sizes]

    # Initialize accumulator from first client
    aggregated = {k: np.zeros_like(v) for k, v in client_weights[0].items()}

    for weight_dict, scale in zip(client_weights, scaling_factors):
        for key in aggregated:
            aggregated[key] += scale * weight_dict[key]

    return aggregated

In [4]:
# Server Class
class FederatedServer:
    def __init__(self, global_model_weights: Dict[str, np.ndarray]):
        """
        Args:
            global_model_weights: initial model weights (state dict as numpy arrays)
        """
        self.global_weights = copy.deepcopy(global_model_weights)
        self.round_history: List[Dict] = []

    # ── Aggregation ──────────────────────────
    def aggregate(
        self,
        client_updates: List[Tuple[Dict[str, np.ndarray], int]]
    ) -> Dict[str, np.ndarray]:
        """
        Run one round of FedAvg aggregation and update the global model.

        Args:
            client_updates: list of (client_weights, num_samples) tuples

        Returns:
            Updated global weights
        """
        weights_list = [w for w, _ in client_updates]
        sizes_list   = [s for _, s in client_updates]

        self.global_weights = federated_average(weights_list, sizes_list)

        self.round_history.append({
            "num_clients": len(client_updates),
            "total_samples": sum(sizes_list),
            "client_sample_sizes": sizes_list,
        })

        return self.global_weights

    # ── Distribution ─────────────────────────
    def get_global_model(self) -> Dict[str, np.ndarray]:
        """Return a deep copy of the current global model for distribution."""
        return copy.deepcopy(self.global_weights)

    def distribute(
        self, client_ids: List[str]
    ) -> Dict[str, Dict[str, np.ndarray]]:
        """
        Distribute the global model to all specified clients.

        Args:
            client_ids: list of client identifiers

        Returns:
            Mapping of client_id → global_weights copy
        """
        return {cid: self.get_global_model() for cid in client_ids}

    # ── Utilities ────────────────────────────
    def summary(self) -> None:
        print(f"Rounds completed : {len(self.round_history)}")
        for i, r in enumerate(self.round_history, 1):
            print(f"  Round {i}: {r['num_clients']} clients, "
                  f"{r['total_samples']} total samples")


In [5]:
#Pytorch Adapter (Optional)
def pytorch_state_dict_to_numpy(state_dict) -> Dict[str, np.ndarray]:
    """Convert a PyTorch OrderedDict state_dict to numpy arrays."""
    return {k: v.cpu().numpy() for k, v in state_dict.items()}

def numpy_to_pytorch_state_dict(numpy_dict: Dict[str, np.ndarray]):
    """Convert numpy weight dict back to a PyTorch-compatible OrderedDict."""
    import torch
    return OrderedDict({k: torch.tensor(v) for k, v in numpy_dict.items()})

In [6]:
if __name__ == "__main__":
    # Simulate initial global model
    init_weights = {
        "fc1.weight": np.random.randn(64, 32).astype(np.float32),
        "fc1.bias":   np.random.randn(64).astype(np.float32),
        "fc2.weight": np.random.randn(10, 64).astype(np.float32),
        "fc2.bias":   np.random.randn(10).astype(np.float32),
    }

    server = FederatedServer(init_weights)
    client_ids = ["client_A", "client_B", "client_C"]

    for fl_round in range(3):
        print(f"\n── Round {fl_round + 1} ──")

        # Server distributes current global model
        local_models = server.distribute(client_ids)

        # Simulate clients training locally (perturb weights slightly)
        client_updates = []
        for cid, weights in local_models.items():
            trained = {k: v + np.random.randn(*v.shape) * 0.01
                       for k, v in weights.items()}
            num_samples = np.random.randint(100, 500)   # simulate varied dataset sizes
            client_updates.append((trained, num_samples))

        # Server aggregates
        new_global = server.aggregate(client_updates)
        print(f"Aggregated. fc1.weight mean: {new_global['fc1.weight'].mean():.6f}")

    server.summary()


── Round 1 ──
Aggregated. fc1.weight mean: -0.006719

── Round 2 ──
Aggregated. fc1.weight mean: -0.006516

── Round 3 ──
Aggregated. fc1.weight mean: -0.006547
Rounds completed : 3
  Round 1: 3 clients, 405 total samples
  Round 2: 3 clients, 1012 total samples
  Round 3: 3 clients, 1154 total samples
